<style>
.info {border-left:5px solid #2563eb;background:#eff6ff;padding:12px 16px;margin:12px 0}
.task {border-left:5px solid #d97706;background:#fffbeb;padding:12px 16px;margin:12px 0}
.success {border-left:5px solid #059669;background:#ecfdf5;padding:12px 16px;margin:12px 0}
.warning {border-left:5px solid #dc2626;background:#fef2f2;padding:12px 16px;margin:12px 0}
</style>

        # Python for Data Science
        ## Week 7 · Session 1 · Relational Databases and SQLAlchemy CRUD

        **Course level:** absolute beginner  
        **Prior learning:** Python functions, pandas DataFrames, and the idea of table keys from Weeks 4–6  
        **Duration:** approximately 90 minutes  
        **Method:** explain → predict → run → change → practise

        A database stores related tables and applies rules to protect their structure. This lesson starts with database vocabulary, then uses an in-memory SQLite database so students can practise safely without installing a database server.

        **Student name:** __________________________ &nbsp;&nbsp; **Date:** __________________


        ## Learning outcomes

        By the end of this session, you should be able to:

        - explain tables, rows, columns, primary keys, and foreign keys
- create a SQLAlchemy engine and inspect its database dialect
- define and create a table with suitable constraints
- insert, select, update, and delete rows inside transactions
- use parameters instead of joining user input into SQL text


## Lesson plan

| Part | Time | Activity |
|---|---:|---|
| Recall | 10 min | Retrieve earlier ideas with a small runnable example |
| Learn | 30 min | Add new concepts in short, explained steps |
| Guided lab | 20 min | Build one complete example with the lecturer |
| Student coding | 25 min | Complete five marked tasks |
| Review | 5 min | Knowledge check and exit ticket |

<div class="info"><strong>How to work:</strong> read the explanation first, predict the output, then run the code with <code>Shift + Enter</code>. Only edit cells marked <strong>Student task</strong>. Ask what every line contributes.</div>


## 1. Database vocabulary from zero

| Term | Plain-language meaning | Example |
|---|---|---|
| Database | An organised collection of related data | a training-centre database |
| Table | Rows about one type of thing | `students` |
| Row | One recorded item | one student |
| Column | One defined attribute | `email` |
| Primary key | A value that uniquely identifies a row | `student_id` |
| Foreign key | A value that refers to another table's primary key | `course_id` |
| Constraint | A rule enforced by the database | email cannot be null |

**SQL** is the language used to communicate with relational databases. **SQLAlchemy** is a Python toolkit that creates connections and safely builds or executes database statements.


## 2. Check the required package

The notebook remains readable if SQLAlchemy is not installed. Run the next cell. If it reports `False`, install the package in the same environment as this notebook, restart the kernel, and run all cells again.

```text
conda install sqlalchemy
```


### Detect and import SQLAlchemy

`find_spec` checks availability without crashing. All database examples below use the same Boolean guard.


In [ ]:
from importlib.util import find_spec

SQLALCHEMY_AVAILABLE = find_spec("sqlalchemy") is not None
print("SQLAlchemy available:", SQLALCHEMY_AVAILABLE)

if SQLALCHEMY_AVAILABLE:
    import sqlalchemy
    from sqlalchemy import Column, Float, Integer, MetaData, String, Table
    from sqlalchemy import create_engine, delete, insert, select, text, update
    print("SQLAlchemy version:", sqlalchemy.__version__)
else:
    print("Install with: conda install sqlalchemy")


## 3. The engine and connection URL

An **engine** manages connections between Python and a database. A database URL describes the dialect, driver, and location.

`sqlite+pysqlite:///:memory:` means:

- use the SQLite dialect;
- use Python's SQLite driver;
- keep this temporary database in memory.

Nothing is written to disk, and closing the notebook loses this practice database.


### Create a safe practice engine

`echo=False` keeps automatic SQL logging quiet. Change it to `True` temporarily if you want to see generated SQL.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    engine = create_engine("sqlite+pysqlite:///:memory:", echo=False)
    print("Dialect:", engine.dialect.name)
    print("Driver:", engine.driver)
else:
    print("Engine example skipped until SQLAlchemy is installed.")


### Open and close a connection

A `with` block closes the connection automatically. `text()` marks a textual SQL statement explicitly.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    with engine.connect() as connection:
        database_name = connection.execute(text("SELECT 'practice database' AS name")).scalar_one()
        print(database_name)
    print("Connection returned safely to the engine.")
else:
    print("Connection example skipped.")


## 4. Define a table schema

A schema describes the columns and their rules before any rows are inserted. Here:

- `student_id` is a unique primary key;
- `name` and `email` are required;
- email is also unique;
- `score` stores a decimal value.


### Describe and create the students table

`MetaData` collects table definitions. `create_all` creates any collected tables that do not already exist.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    metadata = MetaData()
    students_table = Table(
        "students",
        metadata,
        Column("student_id", Integer, primary_key=True),
        Column("name", String(80), nullable=False),
        Column("email", String(120), nullable=False, unique=True),
        Column("score", Float, nullable=False),
    )
    metadata.create_all(engine)
    print("Created table:", students_table.name)
    print("Columns:", [column.name for column in students_table.columns])
else:
    print("Schema example skipped.")


## 5. Create: insert rows

CRUD is a useful memory aid:

| Letter | Operation | SQL idea |
|---|---|---|
| C | Create | `INSERT` |
| R | Read | `SELECT` |
| U | Update | `UPDATE` |
| D | Delete | `DELETE` |

Use `engine.begin()` for a transaction that commits when the block succeeds and rolls back if the block fails.


### Insert several rows in one transaction

Each dictionary maps column names to values. The database checks required and unique constraints.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    initial_students = [
        {"student_id": 1, "name": "Amina", "email": "amina@example.org", "score": 78.0},
        {"student_id": 2, "name": "Ben", "email": "ben@example.org", "score": 64.0},
        {"student_id": 3, "name": "Chen", "email": "chen@example.org", "score": 91.0},
    ]
    with engine.begin() as connection:
        result = connection.execute(insert(students_table), initial_students)
    print("Rows inserted:", result.rowcount)
else:
    print("Insert example skipped.")


## 6. Read: select rows and columns

A query should request only the data needed for its question. Sorting the result makes displayed order intentional.


### Select every student

`.mappings()` exposes each returned row as a mapping of column names to values.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    statement = select(students_table).order_by(students_table.c.student_id)
    with engine.connect() as connection:
        rows = connection.execute(statement).mappings().all()
    for row in rows:
        print(dict(row))
else:
    print("Select example skipped.")


### Filter and select named columns

SQLAlchemy uses column objects in conditions. The generated statement keeps code and values separate.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    high_score_statement = (
        select(students_table.c.name, students_table.c.score)
        .where(students_table.c.score >= 70)
        .order_by(students_table.c.score.desc())
    )
    with engine.connect() as connection:
        high_scores = connection.execute(high_score_statement).all()
    print(high_scores)
else:
    print("Filtered query skipped.")


## 7. Parameterised textual SQL

Never place user input into a SQL string with concatenation or an f-string. That can change the meaning of the query and create a SQL-injection vulnerability. Use named parameters such as `:minimum_score` and pass the value separately.


        ### Pass a value safely

        The SQL structure and the input value travel separately to the database driver.

<div class="warning"><strong>Take care:</strong> Values belong in parameter dictionaries; table and column names need separate allow-list validation because they cannot normally be value parameters.</div>


In [ ]:
if SQLALCHEMY_AVAILABLE:
    minimum_score = 75
    safe_sql = text(
        "SELECT name, score FROM students "
        "WHERE score >= :minimum_score ORDER BY score DESC"
    )
    with engine.connect() as connection:
        safe_rows = connection.execute(
            safe_sql,
            {"minimum_score": minimum_score},
        ).mappings().all()
    print([dict(row) for row in safe_rows])
else:
    print("Parameter example skipped.")


## 8. Update and delete with precise conditions

Before changing stored rows, write and run an equivalent `SELECT` condition. Then check `rowcount`. An omitted `WHERE` condition can affect every row.


### Update one student's score

The primary-key condition targets one row. The transaction commits automatically when the block exits normally.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    update_statement = (
        update(students_table)
        .where(students_table.c.student_id == 2)
        .values(score=69.0)
    )
    with engine.begin() as connection:
        update_result = connection.execute(update_statement)
    print("Rows updated:", update_result.rowcount)
else:
    print("Update example skipped.")


### Delete one precisely identified row

This demonstration adds a temporary row and deletes only that row, keeping the main examples intact.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    with engine.begin() as connection:
        connection.execute(
            insert(students_table),
            {"student_id": 99, "name": "Temporary", "email": "temporary@example.org", "score": 0.0},
        )
        delete_result = connection.execute(
            delete(students_table).where(students_table.c.student_id == 99)
        )
    print("Rows deleted:", delete_result.rowcount)
else:
    print("Delete example skipped.")


## 9. Transactions protect a unit of work

A transaction should make related changes succeed together or fail together. In the next example, a deliberate error occurs after an insert. The context manager rolls the insert back.


### Observe an automatic rollback

After catching the teaching error, a count query proves that student 50 was not saved.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    try:
        with engine.begin() as connection:
            connection.execute(
                insert(students_table),
                {"student_id": 50, "name": "Rollback Test", "email": "rollback@example.org", "score": 80.0},
            )
            raise ValueError("Deliberate teaching error")
    except ValueError as error:
        print("Transaction rolled back:", error)

    check_statement = select(students_table.c.student_id).where(students_table.c.student_id == 50)
    with engine.connect() as connection:
        saved_row = connection.execute(check_statement).first()
    print("Saved row:", saved_row)
else:
    print("Transaction example skipped.")


## Guided lab setup: an inventory table

The next cell creates a separate practice table for all five student tasks. It is provided setup code, so run it before starting the tasks.


### Create the lab database

The starter inventory is deliberately small enough to inspect manually.


In [ ]:
if SQLALCHEMY_AVAILABLE:
    lab_engine = create_engine("sqlite+pysqlite:///:memory:")
    lab_metadata = MetaData()
    inventory_table = Table(
        "inventory",
        lab_metadata,
        Column("item_id", Integer, primary_key=True),
        Column("item_name", String(80), nullable=False),
        Column("quantity", Integer, nullable=False),
        Column("unit_price", Float, nullable=False),
    )
    lab_metadata.create_all(lab_engine)
    print("Lab table ready.")
else:
    print("Install SQLAlchemy before completing the lab.")


## Student coding lab

Complete the five CRUD tasks. Rerun the setup cell if you want to reset the practice database.


        ### Student task 1: Inspect the database connection

        <div class="task"><strong>Your job:</strong> When SQLAlchemy is available, print the lab engine's dialect name and the inventory table's four column names.</div>

        **Check your work**

        - The dialect is `sqlite`.
- Four column names are displayed.
- No private password or connection secret is printed.


In [ ]:
# STUDENT TASK 1
if SQLALCHEMY_AVAILABLE:
    print("Lab database is ready.")
    # Print the dialect and list of column names.
else:
    print("Install SQLAlchemy to complete this task.")


        ### Student task 2: Insert inventory rows

        <div class="task"><strong>Your job:</strong> Insert three supplied dictionaries in one `lab_engine.begin()` transaction and print the affected row count.</div>

        **Check your work**

        - All rows are inserted with one execute call.
- The operation is inside a transaction.
- The printed row count is three.


In [ ]:
# STUDENT TASK 2
inventory_rows = [
    {"item_id": 1, "item_name": "Notebook", "quantity": 20, "unit_price": 3.5},
    {"item_id": 2, "item_name": "Pen", "quantity": 50, "unit_price": 1.2},
    {"item_id": 3, "item_name": "Folder", "quantity": 15, "unit_price": 2.25},
]

if SQLALCHEMY_AVAILABLE:
    # Add one transaction and execute insert(inventory_table).
    print("Rows prepared:", len(inventory_rows))
else:
    print("Insert task skipped.")


        ### Student task 3: Read and filter inventory

        <div class="task"><strong>Your job:</strong> Select item name, quantity, and unit price for items with quantity below 25. Sort from the lowest quantity to the highest and print each returned row.</div>

        **Check your work**

        - Only the three requested columns are returned.
- The threshold is applied in the database query.
- The output order is intentional.


In [ ]:
# STUDENT TASK 3
if SQLALCHEMY_AVAILABLE:
    inventory_query = select(inventory_table).where(inventory_table.c.quantity < 0)
    # Replace the query with the requested columns, condition, and order.
    with lab_engine.connect() as connection:
        inventory_results = connection.execute(inventory_query).mappings().all()
    print([dict(row) for row in inventory_results])
else:
    print("Read task skipped.")


        ### Student task 4: Update with a precise key

        <div class="task"><strong>Your job:</strong> Change the quantity for `item_id` 3 to 25, print the affected row count, and select that row to verify the result.</div>

        **Check your work**

        - The `WHERE` condition uses the primary key.
- Exactly one row is reported as updated.
- A new query verifies the stored value.


In [ ]:
# STUDENT TASK 4
if SQLALCHEMY_AVAILABLE:
    # Build and execute the update inside a transaction.
    verification = select(inventory_table).where(inventory_table.c.item_id == 3)
    with lab_engine.connect() as connection:
        verified_row = connection.execute(verification).mappings().first()
    print("Current row:", dict(verified_row) if verified_row else None)
else:
    print("Update task skipped.")


        ### Student task 5: Delete safely and verify

        <div class="task"><strong>Your job:</strong> Select the row for `item_id` 2 first, delete it with the same condition in a transaction, print row count, and verify that it no longer exists.</div>

        **Check your work**

        - The target is previewed before deletion.
- The delete has a primary-key condition.
- A query proves whether the row remains.



**Optional extension:** Wrap an insert and a deliberate error in one transaction, then prove that the insert was rolled back.


In [ ]:
# STUDENT TASK 5
if SQLALCHEMY_AVAILABLE:
    target_condition = inventory_table.c.item_id == 2
    preview_query = select(inventory_table).where(target_condition)
    with lab_engine.connect() as connection:
        preview_row = connection.execute(preview_query).mappings().first()
    print("Preview:", dict(preview_row) if preview_row else None)
    # Add the delete transaction and a verification query.
else:
    print("Delete task skipped.")


        ## Knowledge check

        Answer these without running new code first.

        1. How do primary and foreign keys differ?
2. What information is contained in a database URL?
3. Why is `engine.begin()` safer for a write transaction?
4. Why should user values be passed as parameters?
5. What should you check before executing an update or delete?


## Exit ticket and preparation

<div class="success"><strong>Exit ticket:</strong> write one idea you can explain, one operation you can code, and one question that remains.</div>

**Before the next session:** Make sure the package check succeeds. The final session uses keys to combine related database tables with joins.
